# ERC circularité · 07 · Composition des pièces d'une nomenclature

À partir d'une liste de pièces (code article et description), ce notebook propose pour chacune sa **classe matière**, celle qui fixe le prix du recycleur et la filière. C'est une entrée du reste du dossier : une classe validée ici devient le type de déchet de la pièce dans les autres notebooks.

**Comment ça marche.** Chaque source (une règle sur la description, une pièce validée qui lui ressemble, le classement antérieur, un document du fabricant, un modèle de langage) *affirme* une classe. Aucune n'est crue sur parole : le **cortex** garde ces affirmations, les compare aux validations faites par une personne, et en déduit ce que vaut chaque type de source. La probabilité d'une classe combine les affirmations selon cette fiabilité.

**Le moins d'appels possible.** La chaîne commence par ce qui est gratuit. Une recherche de documents ou un appel à un modèle n'est lancé que si l'**enjeu** de la pièce (ce que rapporterait au plus de connaître sa classe : masse annuelle × écart entre les prix possibles) dépasse le coût de l'action.

**Confidentialité.** Aucun code article interne ni aucune description ne sort du poste : une recherche ne contient que le fabricant, sa référence et un type générique. Une pièce de conception interne n'est jamais recherchée à l'extérieur.

Vocabulaire : « collecte » désigne ici la recherche de documents sur le web ; « scrapé » reste réservé aux pièces mises au scrap.

In [1]:
import sys
from pathlib import Path

# Le dossier est celui qui contient le code (erc/) : le parent de notebooks/.
dossier = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "erc" / "__init__.py").is_file()), Path.cwd().parent)
sys.path.insert(0, str(dossier))
import erc
from erc import affichage as aff

ctx = erc.executer(jusqua="ligne")

ERC circularité — DÉMONSTRATION (données fictives)


  étape sources       3 artefacts → artefacts/00_sources/


  étape classement   13 artefacts → artefacts/01_classement/


  étape flux         16 artefacts → artefacts/02_flux/


  étape fiabilite    11 artefacts → artefacts/03_fiabilite/


  étape rex          13 artefacts → artefacts/04_rex/


  étape ligne        17 artefacts → artefacts/05_ligne/


In [2]:
from erc import composition
compo = composition.executer(ctx)      # nomenclature du dossier ; à défaut, les pièces consommées dans les extractions
aff.etat(ctx)

DÉMONSTRATION : toutes les données affichées sont fictives. Période : du 01/06/2026 au 01/11/2026, 154 jours extraits. Aucune anomalie de calcul.

## 1. Composition proposée

Une ligne par pièce : la classe la plus probable, sa probabilité, les sources qui l'affirment, et un statut.

| Statut | Sens |
|---|---|
| validée | une personne a fixé la classe : c'est la vérité du dossier |
| acceptée | la probabilité dépasse le seuil d'acceptation ; à contrôler par sondage |
| à valider | des preuves existent mais ne suffisent pas |
| inconnue | aucune source ne s'est prononcée |

**Lecture** : « Enjeu » borne ce que la connaissance de la classe peut rapporter par an. « Prochaine action » dit quoi faire, y compris « aucune » quand apprendre la classe coûterait plus que cela ne rapporte.

**Limite** : en démonstration, la nomenclature, les documents et le modèle sont fictifs.

In [3]:
aff.table(ctx, "composition", "Composition proposée", colonnes=["PN", "Description", "Classe retenue", "Probabilité", "Statut", "Sources", "Enjeu (€/an, borne)", "Appels au modèle", "Prochaine action"])

PN,Description,Classe retenue,Probabilité,Statut,Sources,"Enjeu (€/an, borne)",Appels au modèle,Prochaine action
P-1001,IGBT 400A 1200V,Composant de puissance,"1,00",validée,validation humaine,"0,00",0,aucune
P-1002,IGBT 400A 600V,Composant de puissance,"1,00",acceptée,"analogie, classement antérieur, règle","0,19",0,aucune : contrôle par sondage
P-1003,AL Electrolytic Capacitor 450V 4700uF 105degC,Condensateur,"0,99",acceptée,"classement antérieur, règle",,0,aucune : contrôle par sondage
P-1004,Gate Driver Board,Carte B,"1,00",validée,validation humaine,"0,00",0,aucune
P-1005,Control board assembly,Carte B,"1,00",acceptée,"classement antérieur, règle","1,12",0,aucune : contrôle par sondage
P-1006,Bus resistance 33k ohm 10W,Composant standard,"1,00",acceptée,"classement antérieur, règle","0,32",0,aucune : contrôle par sondage
P-1007,Protection fuse 100A high speed,Composant standard,"1,00",validée,validation humaine,"0,00",0,aucune
P-2001,Power supply board,Carte B,"1,00",validée,validation humaine,"0,00",0,aucune
P-2002,Cooling fan 120mm,Composant standard,"1,00",acceptée,"classement antérieur, règle","1,02",0,aucune : contrôle par sondage
P-3001,Coque boîtier gris,Plastique,"1,00",validée,validation humaine,"0,00",0,aucune


## 2. File de revue

Les pièces à faire valider par une personne, par enjeu décroissant. Remplir « Classe validée » et « Validateur », puis déposer le fichier sous `Acquisition/validations_composition.csv` (colonnes `PN;Classe validée;Validateur;Commentaire`) dans le dossier des extractions : à l'exécution suivante, ces validations entrent dans le cortex et améliorent la fiabilité estimée des sources.

In [4]:
aff.table(ctx, "composition", "File de revue", colonnes=["PN", "Description", "Classe retenue", "Probabilité", "Enjeu (€/an, borne)", "Vaut une revue", "Prochaine action"])

PN,Description,Classe retenue,Probabilité,"Enjeu (€/an, borne)",Vaut une revue,Prochaine action
P-9001,Vis M4 inox,Métal,"0,683",,enjeu inconnu,"revue humaine si la pièce compte : enjeu inconnu (poids, volume ou prix d'une classe manquant)"
P-9002,Radiateur aluminium extrudé,Métal,"0,683",,enjeu inconnu,"revue humaine si la pièce compte : enjeu inconnu (poids, volume ou prix d'une classe manquant)"


## 3. Ce que valent les sources

Pour chaque type de source : l'a priori de départ, qui est une hypothèse de travail, puis ce qui est observé sur les pièces validées. Plus il y a de validations, plus la fiabilité repose sur les faits et moins sur l'hypothèse (« Poids des observations »).

**Lecture** : une source souvent contredite par les validations voit sa fiabilité baisser, et pèse moins dans toutes les probabilités. Un modèle de langage est une source comme une autre.

**Limite** : le calcul suppose que les types de source se trompent indépendamment. Si les règles saisies ici reprennent celles qui ont produit le classement antérieur, les deux comptent en double : ne garder que l'une des deux.

In [5]:
aff.table(ctx, "composition", "Fiabilité des sources")

Type de source,A priori (hypothèse),Affirmations vérifiées,Justes,Fausses,Fiabilité,Poids des observations
analogie,"0,650",0,0,0,"0,650","0,000"
classement antérieur,"0,700",6,6,0,"0,880","0,600"
document,"0,800",0,0,0,"0,800","0,000"
déclaration fabricant,"0,950",0,0,0,"0,950","0,000"
modèle : extraction,"0,600",0,0,0,"0,600","0,000"
modèle : validation,"0,700",0,0,0,"0,700","0,000"
règle,"0,750",6,5,1,"0,800","0,600"


## 4. Journal des acquisitions

Chaque recherche de documents et chaque appel au modèle, avec son coût supposé. Une table courte est le signe que la chaîne s'arrête tôt.

In [6]:
aff.table(ctx, "composition", "Journal des acquisitions")

cree_le,domaine,sujet,action,cout,resultat
2026-11-26T12:06:50,composition,P-5001,collecte,"0,020",1 document(s)
2026-11-26T12:06:50,composition,P-5001,lecture par le modèle,"0,050",Carte C
2026-11-26T12:06:50,composition,P-6001,collecte,"0,020",2 document(s)
2026-11-26T12:06:50,composition,P-6001,lecture par le modèle,"0,050",Carte B
2026-11-26T12:06:50,composition,P-6001,lecture par le modèle,"0,050",Composant de puissance


## 5. Valeur de l'information dans le modèle de ligne

Pour les pièces encore incertaines qui figurent dans le modèle de ligne : ce que rapporterait de connaître leur classe, calculé en résolvant le modèle sous chaque classe possible. « La décision change » dit si la réponse modifierait le plan. Table vide = aucune pièce du modèle n'est incertaine.

La justification est dans `docs/Formal_framework.md`, section 8 (proposition 6).

In [7]:
aff.table(ctx, "composition", "Valeur de l'information")

Type de FRU,Pièce,Paramètre,Borne (€/an),Valeur de l'information (€/an),La décision change,Calcul exact,À acquérir


## 6. Brancher une collecte de documents et un modèle local

Sans collecteur ni modèle, la chaîne s'arrête aux sources gratuites et dit ce qui manque. Les deux se branchent sans modifier le dossier :

```python
from erc import composition

def ma_collecte(requete):
    # requete = {"fabricant": ..., "reference_fabricant": ..., "type": ...} : rien d'interne
    # rendre une liste de documents : {"origine": "site ou URL", "texte": "...", "nature": "document" ou "déclaration fabricant"}
    return []

qwen = composition.ModeleLocal("http://localhost:11434/v1", "nom-du-modele")   # tout serveur compatible « chat completions »
compo = composition.executer(ctx, collecteur=ma_collecte, modele=qwen)
```

Le modèle répond sous contrainte : la classe doit être dans la liste proposée, et la citation qui la justifie doit se trouver mot pour mot dans le document, sinon la réponse est rejetée.

In [8]:
aff.notes(ctx, "composition")

Élément,Valeur
pieces,21
statuts › acceptée,13
statuts › validée,6
statuts › à valider,2
appels au modele,3
cout des acquisitions eur,"0,19"
pieces sans appel payant,19
classes,"Carte B, Carte C, Composant de puissance, Condensateur, Câbles, Composant standard, Plastique, Métal"
seuil acceptation,"0,9"
couts hypotheses de travail › collecte,"0,02"


---\nCe notebook est une entrée du dossier : relancer **00_Demarrage** après avoir validé des pièces pour que leurs classes soient reprises partout.